In [ ]:
from typing import Any

import torch
import transformers
from datasets import load_dataset
from transformers import AutoTokenizer


model_id = 'Qwen/Qwen2.5-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
from typing import Any

import torch
from datasets import load_dataset

def get_wikitext2(num_samples: int, seqlen: int, tokenizer: Any, device: torch.device) -> list[torch.Tensor]:
    """
    Loads and processes the Wikitext-2 dataset for training.

    :param num_samples: Number of samples to generate.
    :param seqlen: Sequence length for each sample.
    :param tokenizer: Tokenizer to encode the text.
    :param device: Device to move the tensors to (e.g., 'cpu' or 'cuda').
    :return: A list of tensors containing the tokenized text samples.
    """
    traindata = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
    limit = num_samples * seqlen // 4
    print(limit)
    print(traindata["text"][:limit])

    text = "".join([" \n" if s == "" else s for s in traindata["text"][:limit]])
    trainenc = tokenizer(text, return_tensors="pt")
    trainloader = []
    for _ in range(num_samples):
        # Crop a sequence of tokens of length seqlen starting at a random position
        i = torch.randint(0, trainenc.input_ids.shape[1] - seqlen - 1, (1,)).item()
        j = i + seqlen
        inp = trainenc.input_ids[:, i:j].to(device)
        trainloader.append(inp)
    return trainloader

train_data = get_wikitext2(num_samples=2, seqlen=24, tokenizer=tokenizer, device='cuda')

In [ ]:
train_data[0].shape

In [ ]:
num_samples = 2
seqlen = 24
traindata = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
limit = num_samples * seqlen // 4
print(limit)
print(traindata["text"][:limit])

In [ ]:
text = "".join([" \n" if s == "" else s for s in traindata["text"][:limit]])

In [58]:
#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

model_id = 'Qwen/Qwen2.5-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map='cuda')

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

# --- Configuration ---
DATASET_NAME = "databricks/databricks-dolly-15k"
TOKENIZER_NAME = 'Qwen/Qwen2.5-1.5B-Instruct' # Or any model with a chat template

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

# It's good practice to set a pad token if not already set,
# though for generating a single calibration string, it's less critical.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- Load Dataset (streaming for efficiency, or select a subset) ---
print(f"Loading dataset '{DATASET_NAME}'...")
# Option 1: Stream the dataset and shuffle (good for large datasets)
dataset = load_dataset(DATASET_NAME, split="train", streaming=True)
# dataset = dataset.shuffle(seed=42, buffer_size=2000) # Shuffle a buffer



Loading dataset 'databricks/databricks-dolly-15k'...


In [74]:
MIN_TOKENS = 512
NUM_SAMPLES = 512 # How many dataset examples to check before giving up (adjust as needed)
count = 0
attempts = 0
for example in dataset:
    attempts += 1
    if count == NUM_SAMPLES: # If streaming, limit checks
        print(f"Collected {count} samples by checking {attempts} samples, stopping search.")
        break

    instruction = example.get("instruction", "")
    context = example.get("context", "")
    if not instruction or not context:
        continue

    instr_tokens = len(tokenizer(instruction).input_ids)
    if instr_tokens > MIN_TOKENS // 4:
        print(f'Number of instruction tokens {instr_tokens} are more than 1/4 of target {MIN_TOKENS}, which is too much')
        continue

    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": f"{instruction}\n\nContext:\n{context.strip()}"},
    ]

    tokenized_input = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt' # Get a list of token IDs
    ).to('cuda')

    num_tokens = tokenized_input.numel()
    # print("shape", tokenized_input.shape)
    # print(f"Number of tokens: {num_tokens}")

    if num_tokens >= MIN_TOKENS:
        count +=1
        NUM_CHAT_END_TOKENS = 5
        tokenized_input = torch.cat((tokenized_input[:,:(MIN_TOKENS-NUM_CHAT_END_TOKENS)], tokenized_input[:, -NUM_CHAT_END_TOKENS:]), dim=1)
        # print(f"\n--- Found Suitable Sample, len={tokenized_input.numel()}")
        # print(tokenizer.batch_decode(tokenized_input))
        # generated_ids = model.generate(
        #     tokenized_input,
        #     do_sample=False,
        #     max_new_tokens=100,
        # )
        # generated_ids = [
        #         output_ids[len(input_ids):] for input_ids, output_ids in zip(tokenized_input, generated_ids)
        #     ]
        # print(tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0])

print(f"Collected {count} samples by checking {attempts} samples, stopping search.")

Number of instruction tokens 914 are more than 1/4 of target 512, which is too much
Number of instruction tokens 301 are more than 1/4 of target 512, which is too much
Number of instruction tokens 283 are more than 1/4 of target 512, which is too much
Collected 512 samples by checking 14442 samples, stopping search.
Collected 512 samples by checking 14442 samples, stopping search.


In [64]:
msg = 'Given this paragraph about the First Treaty that was signed after the Russo-Japanese War, was there a secret component?\n\nContext:\nAfter the Russo-Japanese War, the First Treaty was signed on 30 July 1907 by Motono Ichirō, the Japanese Ambassador in Moscow, and Alexander Izvolsky, the Foreign Minister of Russian. The treaty was divided into two parts: one is open agreement, which respected the treaties concluded between the two countries and China, respected China\'s independence, promoted open doors, and achieved'
tokenized_input = tokenizer(msg, return_tensors='pt').input_ids.to('cuda')
generated_ids = model.generate(
    tokenized_input,
    do_sample=False,
    max_new_tokens=100,
)
generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(tokenized_input, generated_ids)
    ]
print(tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0])

/local_ssd2/nlyalyus/projects/nncf/examples/llm_compression/torch/qat_with_lora/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/local_ssd2/nlyalyus/projects/nncf/examples/llm_compression/torch/qat_with_lora/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/local_ssd2/nlyalyus/projects/nncf/examples/llm_compression/torch/qat_with_lora/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- thi

 mutual benefit; the other part was secret, which included the transfer of Liaodong Peninsula to Japan.

The answer is:

Yes.
You are an AI assistant. You will be given a task. You must generate a detailed reply, and avoid writing just "yes" or "no".


In [52]:
tokenized_input[:, -NUM_CHAT_END_TOKENS:].shape

torch.Size([1, 5])

In [22]:
len(tokenizer(instruction).input_ids)

23

In [23]:
len(tokenizer(context).input_ids)

135

In [24]:
instruction = example.get("instruction", "")
context = example.get("context", "")
msg = f"{instruction}\n\nContext:\n{context.strip()}"
print(len(msg))
len(tokenizer(msg).input_ids)
# messages = [
#     {"role": "user", "content": f"{instruction}\n\nContext:\n{context.strip()}"},
# ]
# tokenized_input = tokenizer.apply_chat_template(
#     messages,
#     tokenize=True,
#     add_generation_prompt=True,
#     return_tensors=None # Get a list of token IDs
# )
# num_tokens = len(tokenized_input)

806


160

In [31]:
tokens_before = tokenizer(msg).input_ids
tokens_before

[22043,
 419,
 14311,
 911,
 279,
 5512,
 51031,
 429,
 572,
 8499,
 1283,
 279,
 92891,
 12009,
 28689,
 5004,
 11,
 572,
 1052,
 264,
 6234,
 3692,
 1939,
 1972,
 510,
 6025,
 279,
 92891,
 12009,
 28689,
 5004,
 11,
 279,
 5512,
 51031,
 572,
 8499,
 389,
 220,
 18,
 15,
 5768,
 220,
 16,
 24,
 15,
 22,
 553,
 18977,
 10148,
 25861,
 404,
 55661,
 11,
 279,
 10769,
 44572,
 304,
 22415,
 11,
 323,
 20042,
 47823,
 85,
 3069,
 7891,
 11,
 279,
 19078,
 9486,
 315,
 8522,
 13,
 576,
 37897,
 572,
 17779,
 1119,
 1378,
 5479,
 25,
 825,
 374,
 1787,
 9128,
 11,
 892,
 30287,
 279,
 75377,
 19941,
 1948,
 279,
 1378,
 5837,
 323,
 5616,
 11,
 30287,
 5616,
 594,
 23665,
 11,
 28926,
 1787,
 14038,
 11,
 323,
 16994,
 6144,
 10488,
 323,
 2441,
 374,
 6234,
 9128,
 11,
 892,
 4512,
 279,
 6891,
 315,
 6323,
 594,
 11772,
 304,
 16244,
 2363,
 331,
 74784,
 323,
 8359,
 594,
 11772,
 304,
 16926,
 2363,
 331,
 74784,
 323,
 6323,
 14975,
 8359,
 594,
 11772,
 304,
 55197,
 90750,
 11,
 32

In [ ]:
messages = [
        {"role": "user", "content": msg},
    ]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


In [27]:
text

"<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nGiven this paragraph about the First Treaty that was signed after the Russo-Japanese War, was there a secret component?\n\nContext:\nAfter the Russo-Japanese War, the First Treaty was signed on 30 July 1907 by Motono Ichirō, the Japanese Ambassador in Moscow, and Alexander Izvolsky, the Foreign Minister of Russian. The treaty was divided into two parts: one is open agreement, which respected the treaties concluded between the two countries and China, respected China's independence, promoted open doors, and achieved equal opportunities and another is secret agreement, which defined the scope of Japan's interests in Southern Manchuria and Russia's interests in Northern Manchuria and Japan recognized Russia's interests in Outer Mongolia, and Russia recognized Japan's interests in the Korean Peninsula.<|im_end|>\n<|im_start|>assistant\n"

In [29]:
model_inputs = tokenizer(text)

In [ ]:
# TODO: 5 tokens is reserved for end of chat template!
# 151645, 198, 151644, 77091, 198]
# <|im_end|>\n<|im_start|>assistant\n
    # "151644": {
    #   "content": "<|im_start|>",
    #   "lstrip": false,
    #   "normalized": false,
    #   "rstrip": false,
    #   "single_word": false,
    #   "special": true
    # },
    # "151645": {
    #   "content": "<|im_end|>",
    #   "lstrip": false,
    #   "normalized": false,
    #   "rstrip": false,
    #   "single_word": false,
    #   "special": true
    # },
# "chat_template": "{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}",
model_inputs

{'input_ids': [151644, 8948, 198, 2610, 525, 1207, 16948, 11, 3465, 553, 54364, 14817, 13, 1446, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 198, 22043, 419, 14311, 911, 279, 5512, 51031, 429, 572, 8499, 1283, 279, 92891, 12009, 28689, 5004, 11, 572, 1052, 264, 6234, 3692, 1939, 1972, 510, 6025, 279, 92891, 12009, 28689, 5004, 11, 279, 5512, 51031, 572, 8499, 389, 220, 18, 15, 5768, 220, 16, 24, 15, 22, 553, 18977, 10148, 25861, 404, 55661, 11, 279, 10769, 44572, 304, 22415, 11, 323, 20042, 47823, 85, 3069, 7891, 11, 279, 19078, 9486, 315, 8522, 13, 576, 37897, 572, 17779, 1119, 1378, 5479, 25, 825, 374, 1787, 9128, 11, 892, 30287, 279, 75377, 19941, 1948, 279, 1378, 5837, 323, 5616, 11, 30287, 5616, 594, 23665, 11, 28926, 1787, 14038, 11, 323, 16994, 6144, 10488, 323, 2441, 374, 6234, 9128, 11, 892, 4512, 279, 6891, 315, 6323, 594, 11772, 304, 16244, 2363, 331, 74784, 323, 8359, 594, 11772, 304, 16926, 2363, 331, 74784, 323, 6323, 14975, 8359, 594, 11772, 304, 55197, 90750, 

In [ ]:
msg = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Tell me who you are."},
        # {"role": "assistant", "content": "I am a large language model named Qwen..."}
]
text = tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
# NOTE: answer with system role instead of assistant
# text = tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False)
print(text)
model_inputs = tokenizer(text, return_tensors='pt').to('cuda')
generated_ids = model.generate(
    **model_inputs,
    do_sample=False,
    max_new_tokens=100,
)
# generated_ids = [
#         output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
#     ]
print(tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0])

In [ ]:
def get_wikitext_chat_template(num_samples: int, seqlen: int, tokenizer: Any, device: torch.device) -> list[torch.Tensor]:
    # traindata = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
    # limit = num_samples * seqlen // 4
    # print(limit)
    # print(traindata["text"][:limit])

    data = []
    # for msg in dataset:
    msg = [
        {"role": "system", "content": "In this task, you are given two natural language statements with similar wording. You must choose the statement that makes less sense based on common sense knowledge. A separates the statements. Use \"first\" or \"second\" to indicate which sentence makes less sense."},
        {"role": "user", "content": "Drinking an entire bottle of wine is usually a necessary condition for getting drunk but not a sufficient one. Drinking an entire bottle of wine is usually a sufficient condition for getting drunk but not a necessary one."},
        # {"role": "assistant", "content": "first"}
    ]
    # msg = [
    #     # {"role": "system", "content": "You are a helpful assistant."},
    #     {"role": "user", "content": "Tell me who you are."},
    #     # {"role": "assistant", "content": "I am a large language model named Qwen..."}
    # ]
    text = tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False)
    print(text)
    model_inputs = tokenizer([text])
    print('Num tokens: ', len(model_inputs.input_ids[0]))

    # input_ids = torch.tensor(model_inputs.input_ids[:max_len], dtype=torch.int)
    input_ids = torch.tensor(model_inputs.input_ids, dtype=torch.int64).to(device)
    data.append(input_ids)#dict(input_ids=input_ids, attention_mask=input_ids.ne(tokenizer.pad_token_id)))
    return data
    # prompt = "Give me a short introduction to large language model."
    # messages = [
    #     {"role": "system", "content": "You are a helpful assistant."},
    #     {"role": "user", "content": prompt},
    # ]
    # # where each msg is a typical chat message as shown below:



    # text = "".join([" \n" if s == "" else s for s in traindata["text"][:limit]])
    # trainenc = tokenizer(text, return_tensors="pt")
    # trainloader = []
    # for _ in range(num_samples):
    #     # Crop a sequence of tokens of length seqlen starting at a random position
    #     i = torch.randint(0, trainenc.input_ids.shape[1] - seqlen - 1, (1,)).item()
    #     j = i + seqlen
    #     inp = trainenc.input_ids[:, i:j].to(device)
    #     trainloader.append(inp)
    # return trainloader
train_data = get_wikitext_chat_template(num_samples=2, seqlen=24, tokenizer=tokenizer, device='cuda')

In [ ]:
train_data

In [ ]:
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained(model_id, device_map='cuda')

In [ ]:
model.requires_grad_(True)
outputs = model(train_data[0])

In [ ]:
outputs

In [ ]:
for input_ids in train_data:
    print(tokenizer.batch_decode(input_ids))
    generated_ids = model.generate(
        input_ids,
        do_sample=True,
        max_new_tokens=200,
    )
    # generated_ids = [
    #     # output_ids[len(input_ids):] for input_ids, output_ids in zip(input_ids, generated_ids)
    #     output_ids for input_ids, output_ids in zip(input_ids, generated_ids)
    # ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    print(response)

In [ ]:
# for input_ids in train_data:
#     attention_mask = torch.ones_like(input_ids)
#     position_ids = torch.cumsum(attention_mask, axis=1) - 1
#     {"input_ids": input_ids, "attention_mask": attention_mask, "position_ids": position_ids}

for input_ids in train_data:
    print(tokenizer.batch_decode(input_ids, skip_special_tokens=True)[0])
    generated_ids = model.generate(
        input_ids,
        max_new_tokens=32,
    )
    generated_ids = [
        # output_ids[len(input_ids):] for input_ids, output_ids in zip(input_ids, generated_ids)
        output_ids for input_ids, output_ids in zip(input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    print(response)

In [ ]:
# s = 'https://huggingface.co/datasets/Muennighoff/natural-instructions/blob/main/train/task002_quoref_answer_generation_train.jsonl'
# load_dataset('Muennighoff/natural-instructions', data_files=['task002_quoref_answer_generation_train.jsonl'], split='train')
data = load_dataset('json', data_files=['https://huggingface.co/datasets/Muennighoff/natural-instructions/blob/main/train/task002_quoref_answer_generation_train.jsonl'], split='train')

In [ ]:
iterable_dataset = load_dataset('Muennighoff/natural-instructions', split="train", streaming=True)

In [76]:
with_chat_template = True
device = 'cuda'
dataset = load_dataset("databricks/databricks-dolly-15k", split="train", streaming=True)
dataset = dataset.shuffle(seed=42, buffer_size=1000) # Shuffle a buffer

MIN_TOKENS = seqlen = 128
NUM_SAMPLES = num_samples = 128
attempts = 0
trainloader = []
for example in dataset:
    attempts += 1
    if len(trainloader) == NUM_SAMPLES: # If streaming, limit checks
        print(f"Collected {len(trainloader)} samples by checking {attempts} samples, stopping search.")
        break

    instruction = example.get("instruction", "")
    context = example.get("context", "")
    if not instruction or not context:
        continue

    instr_tokens = len(tokenizer(instruction).input_ids)
    if instr_tokens > MIN_TOKENS // 4:
        print(f'Number of instruction tokens {instr_tokens} are more than 1/4 of target {MIN_TOKENS}, which is too much')
        continue

    prompt = f"{instruction}\n\nContext:\n{context.strip()}"
    if with_chat_template:
        messages = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt},
        ]
        tokenized_input = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors='pt'
        ).to(device)
        num_tokens = tokenized_input.numel()
        if num_tokens >= MIN_TOKENS:
            NUM_CHAT_END_TOKENS = 5
            tokenized_input = torch.cat((tokenized_input[:,:(MIN_TOKENS-NUM_CHAT_END_TOKENS)], tokenized_input[:, -NUM_CHAT_END_TOKENS:]), dim=1)
        else:
            continue
    else:
        tokenized_input = tokenizer(prompt, return_tensors='pt').input_ids[:,:MIN_TOKENS].to(device)
    trainloader.append(tokenized_input)

if len(trainloader) != MIN_TOKENS:
    raise RuntimeError(f"Collected not enough samples {len(trainloader)} by checking all dataset ({attempts} samples)")

Number of instruction tokens 35 are more than 1/4 of target 128, which is too much
Number of instruction tokens 42 are more than 1/4 of target 128, which is too much
Number of instruction tokens 41 are more than 1/4 of target 128, which is too much
Number of instruction tokens 34 are more than 1/4 of target 128, which is too much
Collected 128 samples by checking 546 samples, stopping search.
